# RAG Chatbot 演示

基于私有文档的多轮问答机器人。支持 PDF/TXT 文件，回答附带引用来源。


## 1. 安装依赖

In [ ]:
# !pip install anthropic chromadb sentence-transformers PyMuPDF python-dotenv

## 2. 加载文档并建立索引

In [ ]:
import sys
sys.path.insert(0, '.')
from main import extract_text, chunk_text, VectorStore

# 创建演示文档
demo_text = """
Artificial intelligence (AI) is intelligence demonstrated by machines.
Machine learning is a subset of AI that learns from data without explicit programming.
Deep learning uses neural networks with many layers to learn complex patterns.
The transformer architecture, introduced in 2017, revolutionized NLP tasks.
Large language models like GPT and Claude are trained on massive text datasets.
Retrieval-Augmented Generation (RAG) combines retrieval with generation for factual accuracy.
"""

with open('/tmp/demo_ai.txt', 'w') as f:
    f.write(demo_text)

text = extract_text('/tmp/demo_ai.txt')
chunks = chunk_text(text, chunk_size=50, overlap=10)
print(f'文档长度: {len(text)} 字符')
print(f'切块数量: {len(chunks)}')
for i, c in enumerate(chunks):
    print(f'  chunk[{i}]: {c[:80]}...')


## 3. 向量索引

In [ ]:
store = VectorStore()
store.add_documents(chunks, source='demo_ai.txt')
print('索引完成')


## 4. 检索演示

In [ ]:
query = 'What is retrieval-augmented generation?'
results = store.search(query, top_k=2)
print(f'问题: {query}\n')
for r in results:
    print(f'  [score={r["score"]:.3f}] {r["text"][:100]}...')


## 5. 完整 RAG 问答

In [ ]:
import os
from dotenv import load_dotenv
import anthropic
from main import generate_answer

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))

questions = [
    'What is deep learning?',
    'How does RAG improve accuracy?',
]

history = []
for q in questions:
    retrieved = store.search(q, top_k=2)
    answer = generate_answer(q, retrieved, history, client)
    print(f'Q: {q}')
    print(f'A: {answer}')
    print()
    history += [{'role': 'user', 'content': q}, {'role': 'assistant', 'content': answer}]


## 6. CLI 使用方式

```bash
python main.py --file your_document.pdf
python main.py --file notes.txt --top-k 5
```
